# Import libraries

In [1]:
import pandas as pd
import numpy as np

# Load the two datasets

In [2]:
import pandas as pd
import numpy as np

df_train = pd.read_csv(
    r"C:\Users\AKSHITHA\Downloads\adult_income.csv"
)

df_test = pd.read_csv(
    r"C:\Users\AKSHITHA\Downloads\adult_test.csv"
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (32561, 15)
Test shape: (16280, 15)


# Add source information

In [3]:
df_train["source_split"] = "train"
df_test["source_split"] = "test"

# Combine both datasets

In [4]:
df = pd.concat([df_train, df_test], ignore_index=True)

print("Combined shape:", df.shape)

Combined shape: (48841, 16)


# Raw Data Inspection

# View first 5 rows

In [5]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source_split
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train


# Check dataset size

In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 48841
Columns: 16


# Check column names

In [7]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education_num',
       'marital_status', 'occupation', 'relationship', 'race', 'sex',
       'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
       'income', 'source_split'],
      dtype='str')

# Check data types

In [8]:
df.dtypes

age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital_status      str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country      str
income              str
source_split        str
dtype: object

# Check duplicate rows

In [9]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 29


# Check ? values and existing missing values

In [10]:
print("Question mark values:")
print((df == "?").sum())

print("\nExisting missing values:")
print(df.isnull().sum())

Question mark values:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
source_split         0
dtype: int64

Existing missing values:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
source_split      0
dtype: int64


# Remove leading/trailing whitespace

In [11]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].str.strip()

C:\Users\AKSHITHA\AppData\Local\Temp\ipykernel_16776\3291089911.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns


# Convert ? and empty strings to missing values

In [12]:
df = df.replace(["?", ""], np.nan)

print("Missing values after conversion:")
print(df.isnull().sum())

Missing values after conversion:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
source_split         0
dtype: int64


# Handle missing categorical values

In [13]:
missing_categorical_cols = [
    "workclass",
    "occupation",
    "native_country"
]

for col in missing_categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [14]:
print(df[missing_categorical_cols].isnull().sum())

workclass         0
occupation        0
native_country    0
dtype: int64


# Numeric Cleaning

# Convert numeric columns

In [15]:
numeric_cols = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check numeric ranges

In [16]:
print("Age:", df["age"].min(), "to", df["age"].max())
print("Education number:", df["education_num"].min(), "to", df["education_num"].max())
print("Hours per week:", df["hours_per_week"].min(), "to", df["hours_per_week"].max())
print("Final weight minimum:", df["fnlwgt"].min())
print("Capital gain minimum:", df["capital_gain"].min())
print("Capital loss minimum:", df["capital_loss"].min())

Age: 17 to 90
Education number: 1 to 16
Hours per week: 1 to 99
Final weight minimum: 12285
Capital gain minimum: 0
Capital loss minimum: 0


# Remove impossible numeric values

In [17]:
before_rows = len(df)

df = df[
    df["age"].between(17, 90) &
    df["education_num"].between(1, 16) &
    df["hours_per_week"].between(1, 99) &
    (df["fnlwgt"] > 0) &
    (df["capital_gain"] >= 0) &
    (df["capital_loss"] >= 0)
].copy()

after_rows = len(df)

print("Rows before validation:", before_rows)
print("Rows after validation:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before validation: 48841
Rows after validation: 48841
Rows removed: 0


# Income Cleaning

# Normalize income labels

In [18]:
df["income"] = (
    df["income"]
    .astype("string")
    .str.strip()
    .str.rstrip(".")
)

print(df["income"].value_counts(dropna=False))

income
<=50K    37154
>50K     11687
Name: count, dtype: Int64


# Create high_income

In [19]:
df["high_income"] = (df["income"] == ">50K").astype(int)

print(df["high_income"].value_counts())

high_income
0    37154
1    11687
Name: count, dtype: int64


# Create age groups

In [20]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[16, 24, 34, 44, 54, 64, 100],
    labels=["17-24", "25-34", "35-44", "45-54", "55-64", "65+"]
)

print(df["age_group"].value_counts().sort_index())

age_group
17-24     8432
25-34    12576
35-44    12193
45-54     8771
55-64     4782
65+       2087
Name: count, dtype: int64


# Create working-hours groups

In [21]:
df["hours_group"] = pd.cut(
    df["hours_per_week"],
    bins=[0, 19, 34, 39, 44, 59, 100],
    labels=["<20", "20-34", "35-39", "40-44", "45-59", "60+"]
)

print(df["hours_group"].value_counts().sort_index())

hours_group
<20       2591
20-34     5804
35-39     3292
40-44    23736
45-59     9565
60+       3853
Name: count, dtype: int64


# Create net capital

In [22]:
df["net_capital"] = (
    df["capital_gain"] - df["capital_loss"]
)

# Final Checks

# Check missing values

In [23]:
print("Final missing values:")
print(df.isnull().sum())

Final missing values:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
source_split      0
high_income       0
age_group         0
hours_group       0
net_capital       0
dtype: int64


# Check duplicates

In [24]:
print("Duplicate rows retained:", df.duplicated().sum())

Duplicate rows retained: 29


# Check final dataset

In [25]:
print("Final dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

Final dataset shape: (48841, 20)

Data types:
age                  int64
workclass              str
fnlwgt               int64
education              str
education_num        int64
marital_status         str
occupation             str
relationship           str
race                   str
sex                    str
capital_gain         int64
capital_loss         int64
hours_per_week       int64
native_country         str
income              string
source_split           str
high_income          int64
age_group         category
hours_group       category
net_capital          int64
dtype: object


# View cleaned dataset

In [26]:

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source_split,high_income,age_group,hours_group,net_capital
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train,0,35-44,40-44,2174
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train,0,45-54,<20,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train,0,35-44,40-44,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train,0,45-54,40-44,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train,0,25-34,40-44,0


# Save cleaned CSV

In [27]:
df.to_csv(
    r"C:\Users\AKSHITHA\Downloads\adult_income.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
